In [3]:
import os
import time
import json

from transformers import AutoTokenizer

In [10]:
from ast import literal_eval
import numpy as np

## Load tokenizer

In [5]:
model_id = "google/gemma-3-270m-it"
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [6]:
tool_calling_tokens = ["<tools>", "</tools>", "<tool_calls>", "</tool_calls>"]
special_tokens = {
    "additional_special_tokens": tool_calling_tokens
}

_ = tokenizer.add_special_tokens(special_tokens)

In [7]:
# save tokenizer
tokenizer.save_pretrained("./toolgemma_tokenizer")

('./toolgemma_tokenizer/tokenizer_config.json',
 './toolgemma_tokenizer/chat_template.jinja',
 './toolgemma_tokenizer/tokenizer.json')

In [1]:
# override chat template in tokenizer folder with updated chat template with tool calling

## Reload tokenizer

In [4]:
tokenizer = AutoTokenizer.from_pretrained("./toolgemma_tokenizer")

## Load dataset

In [5]:
xlam_data_file = "./data/xlam_function_calling_60k.json"

In [7]:
with open(xlam_data_file) as f:
    data = json.load(f)

In [8]:
print("Dataset size: ", len(data))

Dataset size:  60000


## Train-Test-Val split

In [9]:
train_data = data[:40000]
test_data = data[40000:50000]
val_data = data[50000:]

## Utility functions for generating assistant masks

In [11]:
def find_x1_x2_next_x3(arr, x1, x2, x3):
    arr = np.asarray(arr)

    # Locations where x1, x2 occur consecutively
    starts = np.flatnonzero((arr[:-1] == x1) & (arr[1:] == x2))

    # Locations of x3
    x3_idx = np.flatnonzero(arr == x3)

    # For each start, find first x3 occurring after i+1
    pos = np.searchsorted(x3_idx, starts + 2)

    valid = pos < len(x3_idx)

    return list(zip(starts[valid], x3_idx[pos[valid]]))

def get_assistant_mask(input_ids, turn_start_id, assistant_token_id, turn_end_id):
    pairs = find_x1_x2_next_x3(input_ids, turn_start_id, assistant_token_id, turn_end_id)
    n = len(input_ids)
    assistant_mask = np.zeros(n, dtype=int)
    for i, j in pairs:
        assistant_mask[i+2:j+1] = 1

    return assistant_mask

## Map dataset

In [11]:
# convert train, test and val dataset to model consumable dataset